# Random forest Langslingerland Tree Classifieer 

This notebook is used to make a random forest model which uses self made annotations on satellite images in order to predict trees and other vegetation classes.

NOTE! Make sure to run the transform begroeiing_langslingerland_annotations_to_pixels notebook, for transforming annotations to a useable parquet file.

In [1]:
import pandas as pd
import geopandas as gpd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import pprint
import os
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_val_score
from satellite_images_nso_datascience.training.train import train_imbalanced_model, cross_validation_balance_on_date
from satellite_images_nso_datascience.training.utils import get_cross_validation_results_filepath, get_model_filepath
from satellite_images_nso_datascience.training.metric_calculation import calculate_average_metrics, get_metrics
from sklearn.metrics import f1_score
from satellite_images_nso_datascience.other import functions
import satellite_images_nso_datascience.model_metrics.custom_model_metrics as custom_model_metrics
from datetime import datetime

2025/06/20 12:35:38 WARNING mlflow.utils.autologging_utils: You are using an unsupported version of sklearn. If you encounter errors during autologging, try upgrading / downgrading sklearn to a supported version, or try upgrading MLflow.


In [2]:
annotated_pixels_filepath = "C:/repos/satellite-images-nso-datascience/data/annotations/Lansingerland/begroeiing_annotations_pixels_sat_images.parquet"

In [3]:
df = pd.read_parquet(annotated_pixels_filepath)

In [4]:
df['label'].unique()

array(['Gras', 'Anders', 'Boom', 'Lage begroeiing', 'Schaduw gras',
       'Schaduw anders'], dtype=object)

In [5]:
selected_features = ['NIR', 'R', 'G', 'B', 'NDVI', 'NumberOfReturns', 'Z', 'Intensity', 'Classification']
optimal_parameters = {
        "n_estimators": 10, 
        "min_samples_split": 5, 
        "min_samples_leaf": 1,
        "bootstrap": False
}

In [6]:
model = RandomForestClassifier(**optimal_parameters)
scaler = StandardScaler()


In [7]:
# Initialize KFold
kf = KFold(n_splits=4, shuffle=True, random_state=42)

scores = cross_val_score(model, df[selected_features], df["label"], cv=kf)

2025/06/06 14:38:19 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'b2c9cbbb7b1f4067baa64ed05d6b76b2', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/06/06 14:38:19 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\ProgramData\Anaconda3\envs\py310\envs\py312\Lib\site-packages\mlflow\data\digest_utils.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead."
2025/06/06 14:39:22 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '9141c4e69a4741e7ba423a8b7a3cbe71', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/06/06 14:39:22 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\ProgramData\Anaconda3\envs\py310\envs\py312\Lib\site-packages\mlflow\data\digest_utils.py

In [7]:
scores

NameError: name 'scores' is not defined

In [8]:
final_model = RandomForestClassifier(**optimal_parameters)
final_scaler = StandardScaler()

sampling_type_boundary = 898609

train_imbalanced_model(
    X_train=df[selected_features], 
    y_train=df["label"], 
    model=final_model, 
    random_state=1337, 
    sampling_type_boundary=sampling_type_boundary ,
    scaler=final_scaler
)
pprint.pprint(get_metrics(y=df["label"], X=df[selected_features], model=final_model, scaler=final_scaler))

Oversampling to rebalance dataset


2025/06/20 12:35:59 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'b1316e336efc4783b31b890cafa6466b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2025/06/20 12:35:59 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.
2025/06/20 12:35:59 WARNING mlflow.sklearn: Failed to infer model signature: the trained model does not have a `predict` or `transform` function, which is required in order to infer the signature
2025/06/20 12:35:59 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/06/20 12:36:09 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '5b2ba4315c434cfcb2dcc1b70ca6474b', which will track hyperparameters, performance metrics, model

Fitting model


2025/06/20 12:36:36 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'


{'Anders': {'f1-score': 0.9999937166590219,
            'precision': 0.9999968583196408,
            'recall': 0.999990575018143,
            'support': 318303},
 'Boom': {'f1-score': 1.0, 'precision': 1.0, 'recall': 1.0, 'support': 4009},
 'Gras': {'f1-score': 0.9999836994615389,
          'precision': 0.9999673994544842,
          'recall': 1.0,
          'support': 92020},
 'Lage begroeiing': {'f1-score': 1.0,
                     'precision': 1.0,
                     'recall': 1.0,
                     'support': 212},
 'Schaduw anders': {'f1-score': 0.9996335654085746,
                    'precision': 1.0,
                    'recall': 0.9992673992673993,
                    'support': 1365},
 'Schaduw gras': {'f1-score': 1.0,
                  'precision': 1.0,
                  'recall': 1.0,
                  'support': 660}}


In [9]:
final_artefact = {
    "model": final_model,
    "scaler": final_scaler
}

In [10]:
final_model_filepath = "C:/repos/satellite-images-nso-datascience/saved_models/randomforest_begroeiing_herkenning_langslingerland_sat_images"
# Add timestamp to filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
final_model_filename = f"{final_model_filepath}_{timestamp}.sav"

print(f"Saving to {final_model_filename}")
with open(final_model_filename, "wb") as file:
    pickle.dump(final_artefact, file)

Saving to C:/repos/satellite-images-nso-datascience/saved_models/randomforest_begroeiing_herkenning_langslingerland_sat_images_20250620_124227.sav
